# 02 - Preprocessing

Preprocess satellite imagery:
- Cloud masking
- LST retrieval
- Spectral indices
- ERA5 downscaling
- Resampling to common grid

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import rasterio
from pathlib import Path

from src.data.preprocessing import (
    cloud_mask_landsat,
    retrieve_lst,
    compute_spectral_indices,
    downscale_era5,
    resample_raster,
)

In [ ]:
# Load raw Landsat data
data_dir = Path('../data/raw/landsat')
scenes = list(data_dir.glob('*.tif'))
print(f"Found {len(scenes)} scenes")

In [ ]:
# Process each scene
processed_dir = Path('../data/processed')
processed_dir.mkdir(parents=True, exist_ok=True)

for scene in scenes:
    print(f"Processing {scene.name}...")
    
    with rasterio.open(scene) as src:
        data = src.read()
        profile = src.profile.copy()
    
    # Cloud masking
    masked_data = cloud_mask_landsat(data)
    
    # LST retrieval
    lst = retrieve_lst(masked_data)
    
    # Spectral indices
    indices = compute_spectral_indices(masked_data)
    
    # Save
    output_path = processed_dir / f"processed_{scene.stem}.tif"
    profile.update(count=1, dtype='float32')
    
    with rasterio.open(output_path, 'w', **profile) as dst:
        dst.write(lst.astype(np.float32), 1)
    
    print(f"  Saved to {output_path}")

In [ ]:
# ERA5 downscaling
era5_path = Path('../data/raw/era5/era5_2024.nc')
dem_path = Path('../data/raw/dem/hyderabad_dem.tif')

if era5_path.exists() and dem_path.exists():
    downscaled = downscale_era5(
        era5_path=era5_path,
        dem_path=dem_path,
        output_dir=processed_dir / 'era5_downscaled',
        target_resolution=30,
    )
    print(f"Downscaled ERA5 data saved to {processed_dir / 'era5_downscaled'}")

In [ ]:
# Resample all layers to common grid
target_shape = (2160, 2160)  # ~625 km² at 30m

for layer in processed_dir.glob('*.tif'):
    resampled = resample_raster(
        layer,
        target_shape=target_shape,
        method='bilinear',
    )
    print(f"Resampled {layer.name} to {target_shape}")